In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [31]:
import os

print(os.listdir('/kaggle/input'))

['datasets', 'notebooks']


In [32]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    break

/kaggle/input


In [33]:
import os

print(os.listdir('/kaggle/input/datasets'))

['paultimothymooney']


In [34]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/paultimothymooney
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val/PNEUMONIA
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val/NORMAL
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test/PNEUMONIA
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test/NORMAL
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train
/kaggle/input/datasets/paultimothymooney/chest-x

In [37]:
import os

base_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray"

for split in ["train", "test", "val"]:
    print(f"\n{split.upper()} DATA")
    
    normal = len(os.listdir(f"{base_path}/{split}/NORMAL"))
    pneumonia = len(os.listdir(f"{base_path}/{split}/PNEUMONIA"))
    
    print("NORMAL:", normal)
    print("PNEUMONIA:", pneumonia)
    print("TOTAL:", normal + pneumonia)


TRAIN DATA
NORMAL: 1342
PNEUMONIA: 3876
TOTAL: 5218

TEST DATA
NORMAL: 234
PNEUMONIA: 390
TOTAL: 624

VAL DATA
NORMAL: 9
PNEUMONIA: 9
TOTAL: 18


## Experiment 1: Pneumonia Prediction Using 100 Chest X-Ray Images

Performed pneumonia classification on 100 Chest X-Ray images (50 Normal and 50 Pneumonia) using an 80/20 train-test split and compared multiple machine learning and deep learning models.

In [38]:
import os

base_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray"

normal_images = os.listdir(f"{base_path}/train/NORMAL")[:50]
pneumonia_images = os.listdir(f"{base_path}/train/PNEUMONIA")[:50]

print("Normal Images:", len(normal_images))
print("Pneumonia Images:", len(pneumonia_images))
print("Total Images:", len(normal_images) + len(pneumonia_images))

Normal Images: 50
Pneumonia Images: 50
Total Images: 100


In [39]:
import cv2
import os
import numpy as np

images = []
labels = []

# NORMAL = 0
for img_name in os.listdir(f"{base_path}/train/NORMAL")[:50]:
    img_path = os.path.join(f"{base_path}/train/NORMAL", img_name)

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (128, 128))

    images.append(img.flatten())
    labels.append(0)

# PNEUMONIA = 1
for img_name in os.listdir(f"{base_path}/train/PNEUMONIA")[:50]:
    img_path = os.path.join(f"{base_path}/train/PNEUMONIA", img_name)

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (128, 128))

    images.append(img.flatten())
    labels.append(1)

X = np.array(images)
y = np.array(labels)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (100, 16384)
Shape of y: (100,)


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Images:", len(X_train))
print("Testing Images:", len(X_test))

Training Images: 80
Testing Images: 20


### Logistic Regression

Applied Logistic Regression on 100 Chest X-Ray images to classify Normal and Pneumonia cases and evaluated its performance using accuracy and training time.

In [41]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

start_time = time.time()

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

training_time = time.time() - start_time

y_pred = lr.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Logistic Regression Accuracy:", accuracy * 100)
print("Training Time :", round(training_time, 4), "seconds")

Logistic Regression Accuracy: 80.0
Training Time : 1.4575 seconds


### Support Vector Classifier (SVC)

Trained an SVC model on 100 Chest X-Ray images and evaluated its effectiveness in distinguishing between Normal and Pneumonia cases.

In [42]:
import time
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

start_time = time.time()

svc_model = SVC(kernel='linear')

svc_model.fit(X_train, y_train)

svc_time = time.time() - start_time

y_pred_svc = svc_model.predict(X_test)

svc_accuracy = accuracy_score(y_test, y_pred_svc)

print("Accuracy :", round(svc_accuracy * 100, 2), "%")
print("Training Time :", round(svc_time, 4), "seconds")

Accuracy : 85.0 %
Training Time : 0.0424 seconds


### Random Forest

Implemented a Random Forest classifier on the Chest X-Ray dataset and analyzed its classification accuracy and training efficiency.

In [43]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_time = time.time() - start_time

y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Accuracy :", round(rf_accuracy*100,2), "%")
print("Training Time :", round(rf_time,4), "seconds")

Accuracy : 85.0 %
Training Time : 0.4317 seconds


### DenseNet121

Implemented the DenseNet121 deep learning model on 100 Chest X-Ray images to evaluate its performance for pneumonia detection and compare it with traditional machine learning models.

In [44]:
import cv2
import os
import numpy as np

dense_images = []
dense_labels = []

# NORMAL = 0
for img_name in os.listdir(f"{base_path}/train/NORMAL")[:50]:

    img_path = os.path.join(f"{base_path}/train/NORMAL", img_name)

    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))

    dense_images.append(img)
    dense_labels.append(0)

# PNEUMONIA = 1
for img_name in os.listdir(f"{base_path}/train/PNEUMONIA")[:50]:

    img_path = os.path.join(f"{base_path}/train/PNEUMONIA", img_name)

    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))

    dense_images.append(img)
    dense_labels.append(1)

X_dense = np.array(dense_images) / 255.0
y_dense = np.array(dense_labels)

print(X_dense.shape)
print(y_dense.shape)

(100, 224, 224, 3)
(100,)


In [45]:
from sklearn.model_selection import train_test_split

X_train_dense, X_test_dense, y_train_dense, y_test_dense = train_test_split(
    X_dense,
    y_dense,
    test_size=0.20,
    random_state=42,
    stratify=y_dense
)

print(X_train_dense.shape)
print(X_test_dense.shape)

(80, 224, 224, 3)
(20, 224, 224, 3)


In [46]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.19.0


In [66]:
import time
import tensorflow as tf

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

start_time = time.time()

base_model = DenseNet121(
    weights=None,
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_dense,
    y_train_dense,
    epochs=5,
    batch_size=8,
    validation_data=(X_test_dense, y_test_dense),
    verbose=1
)

dense_time = time.time() - start_time

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - accuracy: 0.4750 - loss: 0.6935 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 720ms/step - accuracy: 0.5000 - loss: 0.6931 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 715ms/step - accuracy: 0.5000 - loss: 0.6931 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 727ms/step - accuracy: 0.4250 - loss: 0.6936 - val_accuracy: 0.5000 - val_loss: 0.6930
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 724ms/step - accuracy: 0.4875 - loss: 0.6931 - val_accuracy: 0.5000 - val_loss: 0.6930


In [68]:
loss, dense_accuracy = model.evaluate(
    X_test_dense,
    y_test_dense,
    verbose=0
)

print("Accuracy :", round(dense_accuracy * 100, 2), "%")
print("Training Time :", round(dense_time, 4), "seconds")

Accuracy : 50.0 %
Training Time : 57.3695 seconds


## Model Comparison

The performance of Logistic Regression, SVC, Random Forest, and DenseNet121 was compared using accuracy and training time to identify the most effective approach for pneumonia detection.

In [79]:
print("\nMODEL COMPARISON")
print("---------------------------")

print("Logistic Regression :", lr_accuracy * 100, "%")
print("SVC                 :", svc_accuracy * 100, "%")
print("Random Forest       :", rf_accuracy * 100, "%")
print("DenseNet121         :", dense_accuracy * 100, "%")


MODEL COMPARISON
---------------------------
Logistic Regression : 94.5 %
SVC                 : 94.5 %
Random Forest       : 92.0 %
DenseNet121         : 50.0 %


## Conclusion

Among all evaluated models, SVC and Random Forest achieved the highest accuracy of 85% for pneumonia prediction. SVC required the least training time, making it the most efficient model for the 100-image dataset.

## Experiment 2: Pneumonia Prediction Using 1000 Chest X-Ray Images

Performed pneumonia classification on 1000 Chest X-Ray images (500 Normal and 500 Pneumonia) using an 80/20 train-test split to evaluate model performance.

In [62]:
import cv2
import os
import numpy as np

images = []
labels = []

# NORMAL = 0
for img_name in os.listdir(f"{base_path}/train/NORMAL")[:500]:

    img_path = os.path.join(f"{base_path}/train/NORMAL", img_name)

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (128,128))

    images.append(img.flatten())
    labels.append(0)

# PNEUMONIA = 1
for img_name in os.listdir(f"{base_path}/train/PNEUMONIA")[:500]:

    img_path = os.path.join(f"{base_path}/train/PNEUMONIA", img_name)

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (128,128))

    images.append(img.flatten())
    labels.append(1)

X = np.array(images)
y = np.array(labels)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (1000, 16384)
Shape of y: (1000,)


In [64]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Images:", len(X_train))
print("Testing Images:", len(X_test))

Training Images: 800
Testing Images: 200


### Logistic Regression

Applied Logistic Regression on a dataset of 1000 Chest X-Ray images (500 Normal and 500 Pneumonia) using an 80/20 train-test split to evaluate pneumonia prediction performance.

In [72]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

start_time = time.time()

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

lr_time = time.time() - start_time

y_pred_lr = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("Accuracy :", round(lr_accuracy*100,2), "%")
print("Training Time :", round(lr_time,4), "seconds")

Accuracy : 94.5 %
Training Time : 3.7415 seconds


### Support Vector Classifier (SVC)

Trained an SVC model on 1000 Chest X-Ray images and evaluated its effectiveness in distinguishing between Normal and Pneumonia cases.

In [73]:
import time
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

start_time = time.time()

svc_model = SVC(kernel='linear')

svc_model.fit(X_train, y_train)

svc_time = time.time() - start_time

y_pred_svc = svc_model.predict(X_test)

svc_accuracy = accuracy_score(y_test, y_pred_svc)

print("Accuracy :", round(svc_accuracy*100,2), "%")
print("Training Time :", round(svc_time,4), "seconds")

Accuracy : 94.5 %
Training Time : 1.8769 seconds


### Random Forest

Implemented a Random Forest classifier on the Chest X-Ray dataset and analyzed its classification accuracy and training efficiency.

In [74]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_time = time.time() - start_time

y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Accuracy :", round(rf_accuracy*100,2), "%")
print("Training Time :", round(rf_time,4), "seconds")

Accuracy : 92.0 %
Training Time : 6.3846 seconds


### DenseNet121

Implemented the DenseNet121 deep learning model on 1000 Chest X-Ray images using an 80/20 train-test split to evaluate its performance for pneumonia detection and compare it with traditional machine learning models.

In [76]:
import time
import tensorflow as tf

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

start_time = time.time()

base_model = DenseNet121(
    weights=None,
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_dense,
    y_train_dense,
    epochs=5,
    batch_size=8,
    validation_data=(X_test_dense, y_test_dense),
    verbose=1
)

dense_time = time.time() - start_time

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.5000 - loss: 0.6940 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 749ms/step - accuracy: 0.5000 - loss: 0.6937 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 735ms/step - accuracy: 0.5000 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6933
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 736ms/step - accuracy: 0.5000 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.6933
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 752ms/step - accuracy: 0.5000 - loss: 0.6936 - val_accuracy: 0.5000 - val_loss: 0.6932


In [83]:
loss, dense_accuracy = model.evaluate(
    X_test_dense,
    y_test_dense,
    verbose=0
)

print("Accuracy :", round(dense_accuracy*100,2), "%")
print("Training Time :", round(dense_time,4), "seconds")

Accuracy : 50.0 %
Training Time : 60.571 seconds


**DenseNet121 achieved lower accuracy because the model was trained from scratch without pretrained ImageNet weights and under limited computational resources.**

## Model Comparison (1000 Images)

The performance of Logistic Regression, SVC, Random Forest, and DenseNet121 was evaluated on 1000 Chest X-Ray images using accuracy and training time to identify the most effective model for pneumonia prediction.

In [82]:
print("\nMODEL COMPARISON (1000 IMAGES)")
print("------------------------------------------------")

print("Logistic Regression : Accuracy =", round(lr_accuracy*100,2), "% | Time =", round(lr_time,4), "sec")
print("SVC                 : Accuracy =", round(svc_accuracy*100,2), "% | Time =", round(svc_time,4), "sec")
print("Random Forest       : Accuracy =", round(rf_accuracy*100,2), "% | Time =", round(rf_time,4), "sec")
print("DenseNet121         : Accuracy =", round(dense_accuracy*100,2), "% | Time =", round(dense_time,4), "sec")


MODEL COMPARISON (1000 IMAGES)
------------------------------------------------
Logistic Regression : Accuracy = 94.5 % | Time = 3.7415 sec
SVC                 : Accuracy = 94.5 % | Time = 1.8769 sec
Random Forest       : Accuracy = 92.0 % | Time = 6.3846 sec
DenseNet121         : Accuracy = 50.0 % | Time = 60.571 sec


## Conclusion

Four different models were trained and evaluated for pneumonia prediction using 1000 Chest X-Ray images. The models were compared based on accuracy and training time. Traditional machine learning models demonstrated effective performance with lower computational cost, while DenseNet121 required significantly more training time. The model with the highest accuracy was identified as the most suitable approach for pneumonia detection on this dataset.